# VigilantDrive from-scratch training (Colab T4)
This notebook reads the project and dataset from Google Drive; it does not clone or import a GitHub repository. For quicker Drive upload, put one ZIP named `VigilantDrive_from_scratch.zip` in `MyDrive`; alternatively upload the extracted folder as `MyDrive/VigilantDrive_from_scratch`.
The ZIP must contain the top-level `VigilantDrive_from_scratch/` folder. The legacy checkpoint in `baseline_artifacts/` is kept for reference; it is not assumed compatible with the newly verified FCBAM model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile
DRIVE_ZIP = Path('/content/drive/MyDrive/VigilantDrive_from_scratch.zip')
DRIVE_FOLDER = Path('/content/drive/MyDrive/VigilantDrive_from_scratch')
if DRIVE_ZIP.is_file():
    with zipfile.ZipFile(DRIVE_ZIP) as archive: archive.extractall('/content')
    PROJECT = Path('/content/VigilantDrive_from_scratch')
else:
    PROJECT = DRIVE_FOLDER
assert (PROJECT / 'scripts/train.py').is_file(), f'Project not found or ZIP layout is wrong: {PROJECT}'
assert (PROJECT / 'data/Driver_Drowsiness_YOLO.training_ready/data.yaml').is_file(), 'Dataset YAML missing'
print('Project:', PROJECT)

In [ ]:
import torch, subprocess, sys, os
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU unavailable. In Runtime > Change runtime type select T4 GPU, save, then reconnect and rerun.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
%cd $PROJECT
%pip install -q ultralytics==8.3.161 opencv-python==4.11.0.86 PyYAML==6.0.2
# Colab provides its own CUDA-enabled PyTorch. Do not replace it with the local pinned torch wheel.
import torch
assert torch.cuda.is_available(), 'CUDA became unavailable after dependency setup'

In [ ]:
!python scripts/verify_fcbam.py

## Choose one run
Run baseline and FCBAM separately with matching dataset, image size, epochs, and seed. Batch `-1` asks Ultralytics to choose a memory-aware batch size. Drive-backed runs preserve outputs if the Colab session disconnects, though Drive I/O may slow training.

In [ ]:
DATA = 'data/Driver_Drowsiness_YOLO.training_ready/data.yaml'
RUNS = '/content/drive/MyDrive/VigilantDrive_colab_runs'
EPOCHS = 100
IMGSZ = 640
SEED = 42
RUN_NAME = 'baseline_yolo11s'  # Set to fcbam_yolo11s for the custom architecture
MODEL = 'yolo11s.pt'          # Set to 'configs/yolov11s_fcbam.yaml' for FCBAM
TRANSFER = '--pretrained yolo11s.pt' if MODEL.endswith('_fcbam.yaml') else ''
!python scripts/train.py --data "$DATA" --model "$MODEL" $TRANSFER --epochs $EPOCHS --imgsz $IMGSZ --batch -1 --seed $SEED --name "$RUN_NAME" --project "$RUNS" --device 0 --workers 2

If a run is interrupted, do not rerun the training cell from epoch zero. After reconnecting Drive and rerunning setup, execute the resume cell below with the interrupted run's `weights/last.pt` path. The saved checkpoint restores its training state.

In [ ]:
RESUME_CHECKPOINT = ''  # Example: '/content/drive/MyDrive/VigilantDrive_colab_runs/baseline_yolo11s/weights/last.pt'
if RESUME_CHECKPOINT:
    !python scripts/train.py --resume "$RESUME_CHECKPOINT" --device 0

In [ ]:
BEST = f'{RUNS}/{RUN_NAME}/weights/best.pt'
assert Path(BEST).is_file(), f'Best checkpoint not found: {BEST}'
print('Best checkpoint:', BEST)

In [ ]:
!python scripts/evaluate.py --weights "$BEST" --data "$DATA" --split test --device 0